# Obfuscated Malware Detection - Full Execution Pipeline
This notebook contains the complete workflow for training, evaluating, and visualizing models on the CIC-MalMem-2022 dataset.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from src.preprocess import load_and_preprocess_data

## 1. Load & Preprocess Data

In [ ]:
X_train, X_test, y_train, y_test, feature_names = load_and_preprocess_data('data')
print(f'Training shape: {X_train.shape}, Testing shape: {X_test.shape}')

## 2. Train Models & Evaluate

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    print(f'=== {name} ===')
    print(classification_report(y_test, preds))

## 3. Feature Importance Analysis (XGBoost)

In [ ]:
best_model = models['XGBoost']
importances = best_model.feature_importances_
indices = np.argsort(importances)[::-1][:10]

plt.figure(figsize=(10, 6))
plt.title('Top 10 Important Features (XGBoost)')
plt.bar(range(10), importances[indices], align='center')
plt.xticks(range(10), [feature_names[i] for i in indices], rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 4. Save Best Model

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(best_model, 'models/best_model.pkl')
print('Best model successfully saved to models/best_model.pkl')